# WMT14 English→German MT generation (Gemma 3 **1B**, **no LoRA**)

This notebook loads **Gemma 3 1B IT** as the base model (no adapter) and runs:

- **Base model:** `google/gemma-3-1b-it`
- **Task:** **English → German** translation (strict translation-only output)
- **Dataset:** `wmt14` test split (parallel EN/DE)
- **Batching:** default **20** rows per `generate` call (auto backs off on OOM)
- **Multi‑GPU:** runs **one model replica per GPU** (uses up to 2 GPUs if available)

> **Important (Gemma access):** Gemma models are gated on Hugging Face. Make sure you have accepted the model license on the model page and that your HF token has access.


In [1]:
# Cell 1 — ✅ Dependency setup + fix for Kaggle/TensorFlow protobuf crash
# If you see:
#   AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
# it's often protobuf being too new for the TF/XLA packages on the runtime.
# We pin protobuf==3.20.3 *before* importing anything that might touch TF.

import os, sys, subprocess

# Avoid Transformers importing TF/Flax at all (we only use PyTorch)
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

WORKDIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.makedirs(WORKDIR, exist_ok=True)

def pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    print(" ".join(cmd))
    subprocess.check_call(cmd)

# Ensure packaging exists for version checks
pip_install(["-U", "packaging"])

from packaging import version
import google.protobuf
print("Current protobuf:", google.protobuf.__version__)

# If protobuf is >=6, downgrade to avoid TF/XLA breakage.
if version.parse(google.protobuf.__version__) >= version.parse("6.0.0"):
    pip_install(["-U", "protobuf==3.20.3"])
    print("✅ Installed protobuf==3.20.3. Restarting kernel to apply...")
    os._exit(0)

# Constrain protobuf so later installs don’t upgrade it again.
constraints_path = os.path.join(WORKDIR, "pip_constraints.txt")
with open(constraints_path, "w") as f:
    f.write("protobuf==3.20.3\n")

# HF inference stack (Gemma 3 support is available in modern Transformers releases)
pip_install([
    "-U",
    "-c", constraints_path,
    "transformers>=4.49.0",
    "accelerate>=0.33",
    "peft>=0.12",
    "bitsandbytes>=0.43",
    "datasets>=2.20",
    "huggingface_hub>=0.24",
    "sentencepiece",
    "sacrebleu",
    "pandas",
    "tqdm",
])

print("✅ Dependencies ready")


/usr/bin/python3 -m pip install -q -U packaging
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 2.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain-core 0.3.79 requires packaging<26.0.0,>=23.2.0, but you have packaging 26.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is inc

Current protobuf: 5.29.5
/usr/bin/python3 -m pip install -q -U -c /kaggle/working/pip_constraints.txt transformers>=4.49.0 accelerate>=0.33 peft>=0.12 bitsandbytes>=0.43 datasets>=2.20 huggingface_hub>=0.24 sentencepiece sacrebleu pandas tqdm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.2 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
sentence-transformers 5.1.1 req

In [2]:
# Cell 2 — Environment sanity checks + maximum-fidelity FP32 settings
import platform, torch

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA devices:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f" - GPU{i}: {props.name} | {props.total_memory/1e9:.2f} GB")

# ------------------------------------------------------------------
# 🔒 Maximum-fidelity inference settings
# ------------------------------------------------------------------
# We want TRUE FP32 end-to-end:
# - disable TF32 (tensor cores) which trades a bit of precision for speed
# - avoid any autocast / quantization later in the notebook
torch.set_default_dtype(torch.float32)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    # Guard against "reduced precision reductions" (can affect numerical fidelity)
    if hasattr(torch.backends.cuda.matmul, "allow_fp16_reduced_precision_reduction"):
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    if hasattr(torch.backends.cuda.matmul, "allow_bf16_reduced_precision_reduction"):
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

# Highest-precision matmul preference (PyTorch 2.x)
try:
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass

print("TF32 matmul allowed:", getattr(torch.backends.cuda.matmul, "allow_tf32", None))
print("TF32 cuDNN allowed:", getattr(torch.backends.cudnn, "allow_tf32", None))


Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.8.0+cu126
CUDA available: True
CUDA devices: 2
 - GPU0: Tesla T4 | 15.64 GB
 - GPU1: Tesla T4 | 15.64 GB
TF32 matmul allowed: False
TF32 cuDNN allowed: False


In [3]:
# Cell 3 — Cache locations (Kaggle-friendly)
import os

# Keep downloads in a writable directory
os.environ.setdefault("HF_HOME", os.path.join(WORKDIR, "hf_home"))
os.environ.setdefault("TRANSFORMERS_CACHE", os.path.join(WORKDIR, "hf_home", "transformers"))
os.environ.setdefault("HF_DATASETS_CACHE", os.path.join(WORKDIR, "hf_home", "datasets"))

print("HF_HOME:", os.environ["HF_HOME"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])


HF_HOME: /kaggle/working/hf_home
TRANSFORMERS_CACHE: /kaggle/working/hf_home/transformers
HF_DATASETS_CACHE: /kaggle/working/hf_home/datasets


In [5]:
# Cell 4 — Hugging Face login (recommended: Kaggle Secrets / env var)
#
# Put your token in:
# - Kaggle Secrets as HF_TOKEN, OR
# - environment variable HF_TOKEN / HUGGINGFACE_HUB_TOKEN
#
# NOTE: Do NOT hardcode tokens in notebooks.

import os
from huggingface_hub import login

hf_token = None

# Kaggle Secrets (best option on Kaggle)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# Environment variables (works on Colab/local too)
hf_token = hf_token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not hf_token:
    raise ValueError(
        "HF token not found. Set Kaggle Secret HF_TOKEN or env var HF_TOKEN/HUGGINGFACE_HUB_TOKEN."
    )

login(token=hf_token)
print("✅ Logged into Hugging Face Hub")


✅ Logged into Hugging Face Hub


In [6]:
# Cell 5 — Base model repo ID + fail-fast download (NO LoRA)
from huggingface_hub import hf_hub_download

BASE_MODEL_ID = "google/gemma-3-1b-it"  # Gemma 3 1B instruction-tuned

# Fail-fast: base model reachable
_ = hf_hub_download(repo_id=BASE_MODEL_ID, filename="config.json", token=hf_token)

print("✅ Base model reachable:", BASE_MODEL_ID)
print("✅ LoRA disabled: no adapter will be loaded")


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

Adapter repo file count: 8
Adapter repo files: ['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


adapter_config.json:   0%|          | 0.00/932 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/262M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Base model reachable: google/gemma-3-1b-it
✅ Adapter reachable: gaokerena/amestris_1b-it-dpo-en2du-r32a32
ADAPTER_DIR: /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0
 - adapter_config.json    |     0.00 MB | /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0/adapter_config.json
 - adapter_model.safetensors |   250.25 MB | /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0/adapter_model.safetensors
 - tokenizer.json         |    31.84 MB | /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0/tokenizer.json
 - tokenizer_config.json  |     1.10 MB | /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475

In [7]:
# Cell 6 — Load WMT14 test split and build a DataFrame for EN→DE
from datasets import load_dataset, get_dataset_config_names
import pandas as pd

print("Available WMT14 configs:", get_dataset_config_names("wmt14"))

# WMT14 provides the EN/DE parallel test split under the 'de-en' config.
# Direction is defined by which field you treat as source vs reference.
WMT14_CONFIG = "de-en"
test_ds = load_dataset("wmt14", WMT14_CONFIG, split="test")

rows = []
for i, ex in enumerate(test_ds, start=1):
    rows.append({
        "id": i,
        "english": ex["translation"]["en"],
        "german_ref": ex["translation"]["de"],
    })

df = pd.DataFrame(rows)
print("✅ DataFrame created:", df.shape)
display(df.head(3))


README.md: 0.00B [00:00, ?B/s]

Available WMT14 configs: ['cs-en', 'de-en', 'fr-en', 'hi-en', 'ru-en']


de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

✅ DataFrame created: (3003, 3)


,id,english,german_ref
0,1,Gutach: Increased safety for pedestrians,Gutach: Noch mehr Sicherheit für Fußgänger
1,2,They are not even 100 metres apart: On Tuesday...,Sie stehen keine 100 Meter voneinander entfern...
2,3,Two sets of lights so close to one another: in...,Zwei Anlagen so nah beieinander: Absicht oder ...


## Prompt template

We enforce **translation-only** output (no commentary), and we specify the direction **English → German**.


In [9]:
# Cell 7 — Prompt builder for EN→DE
PROMPT_PREFIX_EXACT = '''You are an expert professional translator (English to German), working on data for fine-tuning a machine learning model.
Translation requirements:

Preserve all meaning, nuance, and details.

Do not skip, summarize, shorten, merge, or add content.

Use clear, natural, fluent, professional standard German.

Preserve factual and contextual accuracy (dates, numbers, names, product names, etc.).

If something is already in German (words, phrases, acronyms, etc.), keep it as-is unless a simple grammatical adjustment is clearly needed.

If a term cannot be translated reliably (e.g., brand name, typo, very local slang, ambiguous proper noun), keep the original term and translate around it.

Do not invent extra information.

Do not add explanations or comments; only provide the direct translation.

Text to translate:
'''

def build_prompt(english_text: str) -> str:
    return f"{PROMPT_PREFIX_EXACT}\n\nEnglish:\n{english_text}\n\nGerman:\n"


In [10]:
# Cell 8 — Tokenizer (✅ ALWAYS from BASE model to prevent tokenizer/bytes mismatch mojibake)
from transformers import AutoTokenizer

# IMPORTANT:
# Adapter repos can include a tokenizer.json that silently differs from the base model tokenizer.
# Decoding generated token IDs with a mismatched tokenizer is a classic way to get "mojibake"
# (e.g., 'zusأ¤tzliche' or 'à¤•à¥...' garbage). For correctness, we ALWAYS use the BASE tokenizer.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token, use_fast=True)
print("✅ Loaded tokenizer from BASE model:", BASE_MODEL_ID)

# Ensure we can pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

PAD_ID = int(tokenizer.pad_token_id)
EOS_ID = int(tokenizer.eos_token_id) if tokenizer.eos_token_id is not None else None

# For decoder-only LMs, left-padding is typically best for batching
tokenizer.padding_side = "left"

def _collect_eos_token_ids(tok) -> list[int]:
    """Return a stable list of EOS token IDs including any Gemma end-of-turn tokens."""
    eos_ids = set()
    if tok.eos_token_id is not None:
        eos_ids.add(int(tok.eos_token_id))

    # Heuristic: include any special token whose string hints it's an end-of-turn marker.
    specials = set()
    for attr in ("additional_special_tokens", "all_special_tokens"):
        val = getattr(tok, attr, None)
        if val:
            specials.update(val)

    for s in specials:
        low = s.lower()
        if ("end_of_turn" in low) or ("endofturn" in low) or (low.endswith("eot")) or ("eot" in low):
            tid = tok.convert_tokens_to_ids(s)
            if isinstance(tid, int) and tid != tok.unk_token_id:
                eos_ids.add(int(tid))

    # Explicit fallbacks (safe if token doesn't exist)
    for s in ("<end_of_turn>", "<|end_of_turn|>", "<eot_id>", "<|eot_id|>"):
        try:
            tid = tok.convert_tokens_to_ids(s)
            if isinstance(tid, int) and tid != tok.unk_token_id:
                eos_ids.add(int(tid))
        except Exception:
            pass

    return sorted(eos_ids)

EOS_IDS = _collect_eos_token_ids(tokenizer)

if not EOS_IDS:
    raise RuntimeError('Failed to determine EOS token IDs for the tokenizer; cannot generate safely.')

print("pad_token_id:", PAD_ID)
print("eos_token_id (primary):", EOS_ID)
print("eos_token_ids (incl end-of-turn):", EOS_IDS)
print("has chat_template:", bool(getattr(tokenizer, "chat_template", None)))


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Loaded tokenizer from BASE model: google/gemma-3-1b-it
pad_token_id: 0
eos_token_id (primary): 1
eos_token_ids (incl end-of-turn): [1, 106]
has chat_template: True


In [11]:
# Cell 9 — Prompt formatting + cleanup helpers + numeric stability probe
import torch

def format_as_chat(user_text: str) -> str:
    if getattr(tokenizer, "chat_template", None):
        msgs = [{"role": "user", "content": user_text}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return user_text

def clean_translation(text: str) -> str:
    t = text.strip()
    for prefix in ["Translation:", "German:", "DE:", "Output:", "Answer:"]:
        if t.startswith(prefix):
            t = t[len(prefix):].strip()
    return t

@torch.inference_mode()
def check_logits_finite(model, user_text: str) -> dict:
    text = format_as_chat(user_text)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model(**enc)
    logits = out.logits[:, -1, :]
    finite = torch.isfinite(logits).all().item()
    max_abs = logits.float().abs().max().item() if finite else float("nan")
    return {"finite": finite, "max_abs": max_abs}

print("✅ Helpers ready")


✅ Helpers ready


In [12]:
# Cell 10 — Attention implementation (safe default)
def pick_attn_impl() -> str:
    return "sdpa"

ATTN_IMPL = pick_attn_impl()
print("Attention implementation:", ATTN_IMPL)


Attention implementation: sdpa


In [13]:
# Cell 11 — Load base model in TRUE FP32 (maximum fidelity; no quantization; no half)
import torch
from transformers import AutoModelForCausalLM

def load_base_fp32(device_id: int):
    """Load Gemma 3 1B base model on a single GPU in pure FP32."""
    kwargs = dict(
        token=hf_token,
        torch_dtype=torch.float32,
        device_map={"": device_id},
        low_cpu_mem_usage=True,
        attn_implementation=ATTN_IMPL,
    )

    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **kwargs)
    m.eval()

    # Hard-force FP32 weights everywhere (base + LN + attention projections, etc.)
    m = m.to(dtype=torch.float32)

    # Config: keep model.config.eos_token_id as a single int; generation uses EOS_IDS list.
    m.config.pad_token_id = PAD_ID
    if EOS_ID is not None:
        m.config.eos_token_id = int(EOS_ID)

    # Also set generation config defaults (can be overridden per-call)
    m.generation_config.pad_token_id = PAD_ID
    m.generation_config.eos_token_id = list(EOS_IDS)

    diag = check_logits_finite(m, "Reply with exactly: OK")
    print(f"[GPU{device_id}] Base FP32 finite logits?: {diag['finite']} | max_abs: {diag['max_abs']:.3e}")
    if not diag["finite"]:
        raise RuntimeError(f"[GPU{device_id}] Base model produced non-finite logits even in FP32.")
    return m

print("✅ FP32 base-loading utilities ready")


✅ FP32 base-loading utilities ready


In [14]:
# Cell 12 — No LoRA: use the base model directly (FP32)
#
# In the original notebook, this cell attached a LoRA adapter using PEFT.
# You asked to disable LoRA while keeping everything else the same.
# So this cell now provides a small helper that simply returns the base model as-is.

import torch

def use_base_model_only(base_m):
    """Return the base model (already FP32) and ensure generation IDs are set."""
    m = base_m.to(dtype=torch.float32)
    m.eval()

    m.config.pad_token_id = PAD_ID
    if EOS_ID is not None:
        m.config.eos_token_id = int(EOS_ID)

    m.generation_config.pad_token_id = PAD_ID
    m.generation_config.eos_token_id = list(EOS_IDS)
    return m

print("✅ Base-only helper ready (FP32, no LoRA)")


✅ LoRA attach helper ready (FP32)


In [15]:
# Cell 13 — Generation config (high-quality deterministic decoding)
from transformers import GenerationConfig

def configure_generation(model, *, num_beams: int = 4):
    """High-quality deterministic generation settings for MT."""
    model.config.use_cache = True
    model.generation_config = GenerationConfig.from_model_config(model.config)

    # Deterministic decoding with beam search for better MT quality
    model.generation_config.do_sample = False
    model.generation_config.num_beams = int(num_beams)
    model.generation_config.early_stopping = True

    # Mild anti-repetition safeguards (helps avoid run-on loops)
    model.generation_config.repetition_penalty = 1.05
    model.generation_config.no_repeat_ngram_size = 3

    # Disable sampling-related fields explicitly
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    model.generation_config.temperature = None
    model.generation_config.typical_p = None

    model.generation_config.pad_token_id = PAD_ID
    model.generation_config.eos_token_id = list(EOS_IDS)
    return model

print("✅ Generation config helper ready")


✅ Generation config helper ready


In [16]:
# Cell 14 — Translator class (one replica per GPU; TRUE FP32 end-to-end)
import torch
from typing import List

class Translator:
    def __init__(self, device_id: int):
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is required for this notebook.")
        self.device_id = int(device_id)
        self.device = torch.device(f"cuda:{self.device_id}")
        torch.cuda.set_device(self.device_id)

        # Base model only (FP32; no LoRA)
        base = load_base_fp32(self.device_id)
        self.base_mode = "fp32"

        model = use_base_model_only(base)
        diag = check_logits_finite(model, "Reply with exactly: OK")
        if not diag["finite"]:
            raise RuntimeError(f"[GPU{self.device_id}] Base model produced non-finite logits even in FP32.")

        self.model = configure_generation(model, num_beams=4)
        self.model.eval()

    @torch.inference_mode()
    def translate_batch_once(self, english_texts: List[str], max_new_tokens: int = 256) -> List[str]:
        torch.cuda.set_device(self.device_id)

        prompts = [format_as_chat(build_prompt(t)) for t in english_texts]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

        # Move tensors to device (keep integer dtypes)
        enc = {k: v.to(self.device) for k, v in enc.items()}

        # IMPORTANT: no autocast, no half, no quantization
        outs = self.model.generate(
            **enc,
            generation_config=self.model.generation_config,
            max_new_tokens=int(max_new_tokens),
            min_new_tokens=1,
            pad_token_id=PAD_ID,
            eos_token_id=list(EOS_IDS),  # include <end_of_turn> etc.
        )

        prompt_len = enc["input_ids"].shape[1]
        gen_ids = outs[:, prompt_len:].to("cpu")

        decoded = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        # Cleanup (reduce peak VRAM)
        del outs, gen_ids, enc

        results: List[str] = []
        for txt in decoded:
            txt = clean_translation(txt)
            results.append(txt if txt else "[EMPTY_OUTPUT]")
        return results

    def translate_batch_safe(self, english_texts: List[str], max_new_tokens: int = 256, target_bs: int = 20) -> List[str]:
        out: List[str] = []
        j = 0
        bs = min(int(target_bs), len(english_texts))
        while j < len(english_texts):
            batch = english_texts[j : j + bs]
            try:
                out.extend(self.translate_batch_once(batch, max_new_tokens=max_new_tokens))
                j += len(batch)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs <= 1:
                    raise
                bs = max(1, bs // 2)
                print(f"⚠️ [GPU{self.device_id}] CUDA OOM. Reducing micro-batch to {bs} and retrying...")
        return out

print("✅ Translator class ready (FP32, no LoRA)")


✅ Translator class ready (FP32)


In [17]:
# Cell 15 — Multi-GPU setup (uses up to 2 GPUs, but works with 1)
from concurrent.futures import ThreadPoolExecutor
import torch

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if NUM_GPUS < 1:
    raise RuntimeError("No CUDA GPUs available.")

WORKER_GPUS = list(range(min(2, NUM_GPUS)))
print("Using GPUs:", WORKER_GPUS)

translators = []
for gid in WORKER_GPUS:
    print(f"\n=== Initializing Translator on GPU{gid} ===")
    t = Translator(gid)
    print(f"✅ GPU{gid} ready (base_mode={t.base_mode})")
    translators.append(t)

print("✅ All translators loaded")


`torch_dtype` is deprecated! Use `dtype` instead!


Using GPUs: [0, 1]

=== Initializing Translator on GPU0 ===


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

[GPU0] Base FP32 finite logits?: True | max_abs: 4.234e+01
✅ Using LoRA from: /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.en2de_lora.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.en2de_lora.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.en2de_lora.weight', 'base_model.model.model.layer

✅ GPU0 ready (base_mode=fp32)

=== Initializing Translator on GPU1 ===


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[GPU1] Base FP32 finite logits?: True | max_abs: 4.234e+01
✅ Using LoRA from: /kaggle/working/hf_home/hub/models--gaokerena--amestris_1b-it-dpo-en2du-r32a32/snapshots/ef465e55382281beda59892a322f5e618fb475e0
✅ GPU1 ready (base_mode=fp32)
✅ All translators loaded


In [18]:
# Cell 16 — Parallel translation helper (splits work across GPUs)
from typing import List
from tqdm.auto import tqdm

def translate_texts_parallel(english_texts: List[str], batch_size: int = 20, max_new_tokens: int = 256, desc: str = "Translating") -> List[str]:
    if not english_texts:
        return []

    batches = []
    for i in range(0, len(english_texts), batch_size):
        batches.append((i, english_texts[i:i+batch_size]))

    results = [None] * len(english_texts)

    with ThreadPoolExecutor(max_workers=len(translators)) as ex:
        futures = []
        for idx, (start, batch) in enumerate(batches):
            t = translators[idx % len(translators)]
            futures.append(ex.submit(t.translate_batch_safe, batch, max_new_tokens=max_new_tokens, target_bs=batch_size))

        for (start, batch), fut in tqdm(zip(batches, futures), total=len(batches), desc=desc, unit="batch"):
            outs = fut.result()
            if len(outs) != len(batch):
                raise RuntimeError("Batch output size mismatch.")
            results[start:start+len(batch)] = outs

    if any(x is None for x in results):
        raise RuntimeError("Some outputs are missing (None).")
    return results


In [19]:
# Cell 17 — DRY TEST (alignment + CSV round-trip)
import os, pandas as pd

DRY_N = 5
dry_df = df.head(DRY_N).copy()

dry_texts = dry_df["english"].tolist()
dry_outs = translate_texts_parallel(dry_texts, batch_size=min(5, len(dry_texts)), max_new_tokens=128, desc="Dry test")

dry_df["mt"] = dry_outs

DRY_OUT = os.path.join(WORKDIR, "dry_en_de_mt.csv")
dry_df.to_csv(DRY_OUT, index=False)

print("✅ Dry test saved:", DRY_OUT)
display(dry_df)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'pad_token_id', 'min_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Dry test:   0%|          | 0/1 [00:00<?, ?batch/s]

✅ Dry test saved: /kaggle/working/dry_en_de_mt.csv


,id,english,german_ref,mt
0,1,Gutach: Increased safety for pedestrians,Gutach: Noch mehr Sicherheit für Fußgänger,Gutachten: Erhöhte Sicherheit für Fußgänger
1,2,They are not even 100 metres apart: On Tuesday...,Sie stehen keine 100 Meter voneinander entfern...,Die Straßenleuchten für Fußgänger in Dorpfarkp...
2,3,Two sets of lights so close to one another: in...,Zwei Anlagen so nah beieinander: Absicht oder ...,"Zwei Gruppen von Lichtern, die sich sehr nahe ..."
3,4,"Yesterday, Gutacht's Mayor gave a clear answer...",Diese Frage hat Gutachs Bürgermeister gestern ...,Gutacht’s Bürgermeister gab gestern eine klare...
4,5,"""At the time, the Town Hall traffic lights wer...","""Die Rathausampel ist damals installiert worde...","""Bei der Zeit wurden die Stadtbäume als Schulw..."


## Full run: chunked CSV output (skip existing chunks)

- `CHUNK_SIZE` controls how many rows per output CSV file  
- `BATCH_SIZE` is the target batch size for generation (**20** by default)  
- Existing chunk files are skipped (useful for resume)


In [ ]:
# Cell 18 — Full WMT14 test translation, chunked output
import os
import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = os.path.join(WORKDIR, "wmt14_en_de_mt_chunks")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Output dir:", OUTPUT_DIR)

CHUNK_SIZE = 400
START_CHUNK_INDEX = 0  # 0-based chunk index; set >0 to resume
BATCH_SIZE = 20
MAX_NEW_TOKENS = 256

def chunk_path(start_id: int, end_id: int) -> str:
    return os.path.join(OUTPUT_DIR, f"wmt14_en_de_mt_{start_id:04d}_{end_id:04d}.csv")

num_rows = len(df)
START_ROW = min(START_CHUNK_INDEX * CHUNK_SIZE, num_rows)

print("Total rows:", num_rows, "| chunk size:", CHUNK_SIZE, "| batch size:", BATCH_SIZE)
print("Starting from row:", START_ROW, f"(chunk #{START_CHUNK_INDEX+1})")
print("Using GPUs:", WORKER_GPUS)

pbar = tqdm(total=num_rows, desc="Translating", unit="sent", initial=START_ROW)

for chunk_start in range(START_ROW, num_rows, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, num_rows)

    start_id = int(df.loc[chunk_start, "id"])
    end_id = int(df.loc[chunk_end - 1, "id"])
    out_file = chunk_path(start_id, end_id)

    if os.path.exists(out_file):
        print("⏭️ Skipping existing:", os.path.basename(out_file))
        pbar.update(chunk_end - chunk_start)
        continue

    chunk_df = df.iloc[chunk_start:chunk_end].copy()
    texts = chunk_df["english"].tolist()

    print(f"\n▶ Processing chunk {start_id}..{end_id} ({len(texts)} rows)")
    mts = translate_texts_parallel(
        texts,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        desc=f"chunk {start_id}-{end_id}"
    )

    if len(mts) != len(chunk_df):
        raise RuntimeError("Translation count mismatch for chunk.")

    chunk_df["mt"] = mts
    chunk_df.to_csv(out_file, index=False)
    print("✅ Saved:", out_file)

    pbar.update(len(texts))
    torch.cuda.empty_cache()

pbar.close()
print("\n✅ Done. Files saved under:", OUTPUT_DIR)


✅ Output dir: /kaggle/working/wmt14_en_de_mt_chunks
Total rows: 3003 | chunk size: 400 | batch size: 20
Starting from row: 0 (chunk #1)
Using GPUs: [0, 1]


Translating:   0%|          | 0/3003 [00:00<?, ?sent/s]


▶ Processing chunk 1..400 (400 rows)


chunk 1-400:   0%|          | 0/20 [00:00<?, ?batch/s]

✅ Saved: /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_0001_0400.csv

▶ Processing chunk 401..800 (400 rows)


chunk 401-800:   0%|          | 0/20 [00:00<?, ?batch/s]

✅ Saved: /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_0401_0800.csv

▶ Processing chunk 801..1200 (400 rows)


chunk 801-1200:   0%|          | 0/20 [00:00<?, ?batch/s]

✅ Saved: /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_0801_1200.csv

▶ Processing chunk 1201..1600 (400 rows)


chunk 1201-1600:   0%|          | 0/20 [00:00<?, ?batch/s]

## Optional: merge chunks into one CSV + compute metrics (BLEU / chrF)

If you ran the full translation, this merges all chunk CSVs and computes corpus metrics with **sacrebleu**.


In [21]:
# Cell 19 — Merge chunks and compute corpus metrics
import glob, os
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "wmt14_en_de_mt_*.csv")))
print("Chunk files:", len(chunk_files))

if not chunk_files:
    raise FileNotFoundError("No chunk files found. Run Cell 18 first.")

merged = pd.concat([pd.read_csv(f) for f in chunk_files], ignore_index=True)
merged = merged.sort_values("id").reset_index(drop=True)

MERGED_PATH = os.path.join(OUTPUT_DIR, "wmt14_en_de_mt_merged.csv")
merged.to_csv(MERGED_PATH, index=False)
print("✅ Merged CSV saved:", MERGED_PATH)
display(merged.head(3))

refs = [merged["german_ref"].astype(str).tolist()]
hyps = merged["mt"].astype(str).tolist()

bleu = BLEU().corpus_score(hyps, refs)
chrf = CHRF().corpus_score(hyps, refs)

print("\n📊 Metrics on WMT14 test (EN→DE)")
print("BLEU:", bleu)
print("chrF:", chrf)


Chunk files: 8
✅ Merged CSV saved: /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_merged.csv


,id,english,german_ref,mt
0,1,Gutach: Increased safety for pedestrians,Gutach: Noch mehr Sicherheit für Fußgänger,Gutachten: Erhöhte Sicherheit für Fußgänger
1,2,They are not even 100 metres apart: On Tuesday...,Sie stehen keine 100 Meter voneinander entfern...,Die Straßenleuchten für Fußgänger in Dorpfarkp...
2,3,Two sets of lights so close to one another: in...,Zwei Anlagen so nah beieinander: Absicht oder ...,"Zwei Gruppen von Lichtern, die sich sehr nahe ..."



📊 Metrics on WMT14 test (EN→DE)
BLEU: BLEU = 15.01 46.9/20.2/10.4/5.7 (BP = 0.977 ratio = 0.977 hyp_len = 61232 ref_len = 62688)
chrF: chrF2 = 47.57


In [31]:
# Cell: Compute COMET (per-segment + system average) for your merged EN→DE outputs and write a scored CSV
# Uses your notebook columns:
#   src = english
#   mt  = mt
#   ref = german_ref
# Adds: comet_score
# Appends summary row (id="__SYSTEM_AVG__") if `id` column exists

import os
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

import pandas as pd
import torch

# Avoid tokenizers fork warnings / deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# Resolve merged CSV path (from your prior cells)
# ----------------------------
def _resolve_merged_csv() -> str:
    if "MERGED_PATH" in globals() and isinstance(globals()["MERGED_PATH"], str):
        p = globals()["MERGED_PATH"]
        if os.path.isfile(p):
            return p

    if "merged" in globals() and isinstance(globals()["merged"], pd.DataFrame):
        workdir = globals().get("WORKDIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
        outdir = globals().get("OUTPUT_DIR", os.path.join(workdir, "wmt14_en_de_mt_chunks"))
        os.makedirs(outdir, exist_ok=True)
        p = os.path.join(outdir, "wmt14_en_de_mt_merged.csv")
        globals()["merged"].to_csv(p, index=False, encoding="utf-8")
        return p

    workdir = globals().get("WORKDIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
    outdir = globals().get("OUTPUT_DIR", os.path.join(workdir, "wmt14_en_de_mt_chunks"))
    p = os.path.join(outdir, "wmt14_en_de_mt_merged.csv")
    if os.path.isfile(p):
        return p

    raise FileNotFoundError("Could not locate merged CSV. Run the merge cell first (producing `merged` or `MERGED_PATH`).")

MERGED_CSV = os.path.abspath(_resolve_merged_csv())
OUT_DIR = os.path.dirname(MERGED_CSV)
COMET_SCORED_CSV = os.path.splitext(MERGED_CSV)[0] + "_with_comet.csv"

print("📄 MERGED_CSV :", MERGED_CSV)
print("📝 OUTPUT_CSV :", COMET_SCORED_CSV)

# ----------------------------
# Import COMET (no installs here)
# ----------------------------
try:
    from comet import download_model, load_from_checkpoint  # type: ignore
except Exception as e:
    raise RuntimeError(
        "COMET is not importable in this runtime. "
        "Do NOT pip install here (your env/disk is already constrained).\n"
        f"Import error: {e}"
    )

# ----------------------------
# Patch #1: Transformers>=5 enforces ModelOutput subclasses to be @dataclass.
# COMET's Prediction subclasses ModelOutput but isn't a dataclass -> crash.
# We replace it with a dataclass-compatible drop-in.
# ----------------------------
def _patch_comet_prediction_dataclass() -> bool:
    try:
        from transformers.utils.generic import ModelOutput  # transformers>=4
    except Exception:
        return False

    @dataclass
    class _Prediction(ModelOutput):
        score: Any = None
        scores: Any = None
        system_score: Any = None

    patched_any = False
    try:
        import comet.models.utils as cu  # type: ignore
        cu.Prediction = _Prediction  # type: ignore
        patched_any = True
    except Exception:
        pass

    # Patch already-imported module references (COMET imports Prediction into these modules)
    for mod_name in [
        "comet.models.base",
        "comet.models.regression.regression_metric",
        "comet.models.ranking.ranking_metric",
        "comet.models.multitask.unified_metric",
    ]:
        try:
            mod = __import__(mod_name, fromlist=["Prediction"])
            setattr(mod, "Prediction", _Prediction)
            patched_any = True
        except Exception:
            pass

    return patched_any

patched_pred = _patch_comet_prediction_dataclass()
print("🩹 Patched COMET Prediction dataclass:", "YES" if patched_pred else "NO")

# ----------------------------
# Patch #2: Transformers>=5 model outputs changed; COMET encoder code can unpack wrong shapes.
# We patch XLM-R + BERT encoder forward() to return the dict shape COMET expects.
# ----------------------------
def _extract_last_hidden_and_layers(outputs: Any) -> Tuple[torch.Tensor, Any]:
    # return_dict=True
    if hasattr(outputs, "last_hidden_state"):
        return outputs.last_hidden_state, getattr(outputs, "hidden_states", None)

    # tuple/list fallback
    if isinstance(outputs, (tuple, list)):
        if len(outputs) >= 3:
            return outputs[0], outputs[2]
        if len(outputs) == 2:
            return outputs[0], outputs[1]
        if len(outputs) == 1:
            return outputs[0], None

    raise TypeError(f"Unexpected HF model output type: {type(outputs)}")

def _patch_comet_encoders_tf5() -> List[str]:
    patched: List[str] = []

    # XLM-R encoder used by Unbabel/wmt22-comet-da
    try:
        import comet.encoders.xlmr as xlmr  # type: ignore
        if hasattr(xlmr, "XLMREncoder"):
            def _xlmr_forward(self, input_ids, attention_mask, **kwargs):
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    return_dict=True,
                )
                last_hidden, all_layers = _extract_last_hidden_and_layers(outputs)
                return {
                    "sentemb": last_hidden[:, 0, :],
                    "wordemb": last_hidden,
                    "all_layers": all_layers,
                    "attention_mask": attention_mask,
                }
            xlmr.XLMREncoder.forward = _xlmr_forward  # type: ignore
            patched.append("XLMREncoder")
    except Exception:
        pass

    # BERT encoder (future-proof)
    try:
        import comet.encoders.bert as bert  # type: ignore
        if hasattr(bert, "BERTEncoder"):
            def _bert_forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    output_hidden_states=True,
                    return_dict=True,
                )
                last_hidden, all_layers = _extract_last_hidden_and_layers(outputs)
                return {
                    "sentemb": last_hidden[:, 0, :],
                    "wordemb": last_hidden,
                    "all_layers": all_layers,
                    "attention_mask": attention_mask,
                }
            bert.BERTEncoder.forward = _bert_forward  # type: ignore
            patched.append("BERTEncoder")
    except Exception:
        pass

    return patched

patched_enc = _patch_comet_encoders_tf5()
print("🩹 Patched COMET encoders:", ", ".join(patched_enc) if patched_enc else "(none)")

# ----------------------------
# Load merged CSV & validate schema
# ----------------------------
SRC_COL = "english"
REF_COL = "german_ref"
MT_COL  = "mt"
ID_COL  = "id"
SUMMARY_ID = "__SYSTEM_AVG__"

df = pd.read_csv(MERGED_CSV)
missing = {SRC_COL, REF_COL, MT_COL} - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns in merged CSV: {missing}. Found: {list(df.columns)}")

# Drop previous summary row if present (avoid double counting)
if ID_COL in df.columns:
    df = df[df[ID_COL].astype(str) != SUMMARY_ID].reset_index(drop=True)

# Normalize to strings
for c in (SRC_COL, REF_COL, MT_COL):
    df[c] = df[c].fillna("").astype(str)

# Build COMET data: src=english, mt=mt, ref=german_ref
comet_data: List[Dict[str, str]] = [
    {"src": s, "mt": m, "ref": r}
    for s, m, r in zip(df[SRC_COL].tolist(), df[MT_COL].tolist(), df[REF_COL].tolist())
]

# ----------------------------
# Load COMET model + score
# ----------------------------
COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"
COMET_BATCH_SIZE = 16

# Keep math strict (doesn't change COMET much, but avoids TF32)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
try:
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass

print(f"⬇️  Downloading/loading COMET model: {COMET_MODEL_NAME}")
model_path = download_model(COMET_MODEL_NAME)
comet_model: Any = load_from_checkpoint(model_path)
comet_model.eval()

use_gpu = torch.cuda.is_available()
gpus = 1 if use_gpu else 0
accelerator = "gpu" if use_gpu else "cpu"
if use_gpu:
    print(f"🖥️  CUDA available: using GPU -> {torch.cuda.get_device_name(0)}")
else:
    print("🖥️  CUDA not available: running on CPU")

print("🧮 Computing COMET scores (per-segment + system average)...")
with torch.inference_mode():
    pred_out = comet_model.predict(
        comet_data,
        batch_size=COMET_BATCH_SIZE,
        gpus=gpus,
        accelerator=accelerator,
        num_workers=0,    # IMPORTANT: prevent DataLoader fork issues with tokenizers
        progress_bar=True,
    )

# Extract scores
if hasattr(pred_out, "scores") and hasattr(pred_out, "system_score"):
    comet_scores = [float(s) for s in pred_out.scores]
    comet_system = float(pred_out.system_score)
elif isinstance(pred_out, dict) and "scores" in pred_out and "system_score" in pred_out:
    comet_scores = [float(s) for s in pred_out["scores"]]
    comet_system = float(pred_out["system_score"])
else:
    raise RuntimeError(f"Unexpected COMET output type: {type(pred_out)}; available attrs: {dir(pred_out)}")

if len(comet_scores) != len(df):
    raise RuntimeError(f"COMET score count mismatch: got {len(comet_scores)} scores for {len(df)} rows.")

comet_mean = float(sum(comet_scores) / len(comet_scores)) if comet_scores else float("nan")

# ----------------------------
# Attach scores + append summary row + save
# ----------------------------
scored_df = df.copy()
scored_df["comet_score"] = comet_scores

summary_row = {col: "" for col in scored_df.columns}
if ID_COL in scored_df.columns:
    summary_row[ID_COL] = SUMMARY_ID
summary_row["comet_score"] = comet_system

scored_df = pd.concat([scored_df, pd.DataFrame([summary_row])], ignore_index=True)
scored_df.to_csv(COMET_SCORED_CSV, index=False, encoding="utf-8")

print(f"\n✅ COMET-scored CSV written to: {COMET_SCORED_CSV}")
print(f"📊 mean(COMET per-entry): {comet_mean:.6f}")
print(f"📊 COMET system_score  : {comet_system:.6f}")

display(scored_df.head(5))


📄 MERGED_CSV : /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_merged.csv
📝 OUTPUT_CSV : /kaggle/working/wmt14_en_de_mt_chunks/wmt14_en_de_mt_merged_with_comet.csv
🩹 Patched COMET Prediction dataclass: YES
🩹 Patched COMET encoders: XLMREncoder, BERTEncoder
⬇️  Downloading/loading COMET model: Unbabel/wmt22-comet-da


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint hf_home/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Predicting DataLoader 0:   0%|          | 0/188 [08:16<?, ?it/s]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU avai

🖥️  CUDA available: using GPU -> Tesla T4
🧮 Computing COMET scores (per-segment + system average)...


Predicting DataLoader 0: 100%|██████████| 188/188 [01:33<00:00,  2.02it/s]


OSError: [Errno 28] No space left on device